# A LoKI batch from a table

I(Q) of several samples, with the runs of each sample as a row of a table.
Every step takes and returns a pandas `DataFrame`, and each is also a subcommand of `essapps-batch`, shown at the end:

```
files in a folder ──> table of files ──> table of rows ──> reduced ──> exported
     scan_folder        (add roles)       pair_by_sample   reduce_table  export_table
```

[packages/essapps/docs/batch.md](../../essapps/docs/batch.md) states the rules: what a cell is, what is refused, and what the workspace keeps.
[loki-reduction.ipynb](loki-reduction.ipynb) explains the reduction itself.

## A workspace in a folder

The workspace is a folder that holds the files a request may name, and what the workspace keeps: its history and the outputs that were persisted.
Here it is a temporary folder with links to the runs of the LoKI detector test at Larmor, which the first cell downloads if they are missing.

In [ ]:
import tempfile
from pathlib import Path

import pandas as pd
import scipp as sc
from ess.loki import data

from ess.apps import Template, file
from ess.apps.batch import file_infos, tables
from ess.apps.batch.export import export_table
from ess.apps.folder import open_folder, scan_folder
from ess.instruments.loki.batch import pair_by_sample
from ess.instruments.loki.bindings import BINDINGS
from ess.instruments.loki.files import read_metadata
from ess.instruments.loki.specs import BEAM_CENTRE, CALIBRATED_DETECTOR, IOFQ, MASKS

downloaded = [
    data.loki_tutorial_run_60392(),
    data.loki_tutorial_background_run_60393(),
    data.loki_tutorial_sample_run_60339(),
    data.loki_tutorial_isis_polymer_sample_run(),
    data.loki_tutorial_isis_polymer_transmission_run(),
    data.loki_tutorial_agbeh_sample_run(),
    data.loki_tutorial_agbeh_transmission_run(),
    data.loki_tutorial_porous_silica_sample_run(),
    data.loki_tutorial_porous_silica_transmission_run(),
    data.loki_tutorial_direct_beam_all_pixels(),
    data.loki_tutorial_mask_filenames()[0],
]
workspace = Path(tempfile.mkdtemp())
(workspace / 'files').mkdir()
for path in map(Path, downloaded):
    (workspace / 'files' / path.name).symlink_to(path)

## The table of files

`scan_folder` asks `read_metadata` of `ess.instruments.loki` what each file says of itself: its kind, and for a run its number and title.

In [ ]:
files = pd.DataFrame(scan_folder(workspace, read_metadata)).fillna('')
files

No file says what a run was measured for, so a person adds the role of each run and the name of its sample.
The direct beam needs its role too, since the field of I(Q) that takes it requires it.
Run 60392 is the can's transmission run and the empty beam, so it gets no role and fits both uses.

In [ ]:
measured = {
    '60339': ('sample', 'ISIS polymer, second run'),
    '60395': ('sample', 'ISIS polymer'),
    '60394': ('transmission', 'ISIS polymer'),
    '60387': ('sample', 'AgBeh'),
    '60386': ('transmission', 'AgBeh'),
    '60385': ('sample', 'porous silica'),
    '60384': ('transmission', 'porous silica'),
    '60393': ('background', ''),
}
files['role'] = [measured.get(run, ('', ''))[0] for run in files['run']]
files['sample'] = [measured.get(run, ('', ''))[1] for run in files['run']]
files.loc[files['kind'] == 'calibration', 'role'] = 'direct_beam'
files.to_csv(workspace / 'files.csv', index=False)
files.drop(columns='title')

In [ ]:
client = open_folder(workspace, BINDINGS)

## The table of rows

`pair_by_sample` makes a row for each sample run, with the transmission run of the same sample.
A cell names a file as the workspace identifies it, here by its path.

In [ ]:
table = tables.table_of(pair_by_sample(file_infos(client)))
table.assign(
    **{
        c: table[c].str.rsplit('/', n=1).str[-1]
        for c in ('sample', 'sample_transmission')
    }
)

Run 60339 has a name of its own, so it found no transmission run, and its cell is empty.
A table is edited as any `DataFrame`. A whole number in a file column names a run.

In [ ]:
table.loc[table['key'] == '60339', 'sample_transmission'] = '60394'

## What every row shares

The detector, the masks, and the beam centre are records that the batch reads by reference ([loki-reduction.ipynb](loki-reduction.ipynb)).
The template holds them with the other settings. Its blanks are the columns of the table.

In [ ]:
mask_file = next(workspace.glob('files/*.xml'))
direct_beam = next(workspace.glob('files/direct-beam*'))

detector = client.submit(CALIBRATED_DETECTOR, params={'run': file(run=60339)})
masks = client.submit(
    MASKS,
    params={'detector': detector.ref('detector'), 'mask_files': [file(path=mask_file)]},
)
centre = client.submit(
    BEAM_CENTRE,
    params={'detector': detector.ref('detector'), 'masks': masks.ref('masks')},
)
template = Template(
    IOFQ,
    params={
        'can': file(run=60393),
        'can_transmission': file(run=60392),
        'empty_beam': file(run=60392),
        'direct_beam': file(path=direct_beam),
        'masks': masks.ref('masks'),
        'beam_centre': centre.ref('centre'),
        'wavelength': {'start': 1.0, 'stop': 13.0, 'num_bins': 200},
        'q': {'start': 0.01, 'stop': 0.3, 'num_bins': 100},
    },
    blanks=('sample', 'sample_transmission'),
)

## Reduce

`reduce_table` submits a request for each row, waits, and returns the table with what came of each row.
With `staged=True` the rows go through a stage, which reduces the can once.

In [ ]:
done = tables.reduce_table(client, template, table, label='iofq', staged=True)
done[['key', 'name', 'status', 'record', 'failure']]

The `record` column identifies the result of each row, so the outputs are read by it.

In [ ]:
records = client.records(ids=list(done['record']))
sc.plot(
    {
        name: client.output(record, 'iofq').intensity
        for name, record in zip(done['name'], records, strict=True)
    },
    norm='log',
    title='I(Q) per sample',
)

## Export

`export_table` writes the outputs of each completed row to a folder named by its key, and returns the table with the path of each file.
The files are what `scipp.io.load_hdf5` reads, and `_record.json` names every file and value they came from.

In [ ]:
exported = export_table(client, done, workspace / 'reduced')
exported.assign(
    **{
        c: exported[c].str.removeprefix(f'{workspace}/')
        for c in ('iofq', 'transmission')
    }
)[['key', 'name', 'status', 'iofq', 'transmission']]

In [ ]:
sc.io.load_hdf5(exported['iofq'][0])

In [ ]:
client.close()

## The same from the command line

Each step is a subcommand, and the tables are files. The template is a YAML file, which names the records of the steps before it by their labels:

```sh
B="essapps-batch --bind ess.instruments.loki.bindings:BINDINGS"
essapps-batch files ess.instruments.loki.files:read_metadata -o files.csv   # then add the roles
$B rows ess.instruments.loki.batch:pair_by_sample -o runs.csv               # then edit the rows
$B reduce detector.yaml --label detector
$B reduce masks.yaml --label masks
$B reduce centre.yaml --label centre
$B reduce iofq.yaml runs.csv --label iofq --staged -o done.csv
$B export done.csv reduced -o exported.csv
```

```yaml
# iofq.yaml
spec: {name: loki-iofq, version: 1}
blanks: [sample, sample_transmission]
params:
  can: 60393
  can_transmission: 60392
  empty_beam: 60392
  direct_beam: files/direct-beam-loki-all-pixels.h5
  masks: {'@latest': masks, output: masks}
  beam_centre: {'@latest': centre, output: centre}
  wavelength: {start: 1.0, stop: 13.0, num_bins: 200}
  q: {start: 0.01, stop: 0.3, num_bins: 100}
```